<a href="https://colab.research.google.com/github/Jamilly-Melo/projeto-2-SBC/blob/main/projeto_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q scikit-fuzzy

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [ ]:
# Entradas

# Umidade do ambiente (%)
umidade = ctrl.Antecedent(np.arange(0, 101, 1), 'umidade')

# Temperatura ambiente (°C)
temperatura = ctrl.Antecedent(np.arange(0, 41, 1), 'temperatura')


# saída

# Risco de fungos (%)
risco_fungos = ctrl.Consequent(np.arange(0, 101, 1), 'risco_fungos')

In [ ]:
# Umidade
umidade['baixa'] = fuzz.trimf(umidade.universe, [0, 0, 40])
umidade['media'] = fuzz.trimf(umidade.universe, [20, 50, 80])
umidade['alta'] = fuzz.trimf(umidade.universe, [60, 100, 100])

# Temperatura
temperatura['baixa'] = fuzz.trimf(temperatura.universe, [0, 0, 18])
temperatura['ideal'] = fuzz.trimf(temperatura.universe, [15, 25, 35])
temperatura['alta'] = fuzz.trimf(temperatura.universe, [30, 40, 40])

# risco de fungos
risco_fungos['baixo'] = fuzz.trimf(risco_fungos.universe, [0, 0, 40])
risco_fungos['medio'] = fuzz.trimf(risco_fungos.universe, [20, 50, 80])
risco_fungos['alto'] = fuzz.trimf(risco_fungos.universe, [60, 100, 100])

In [ ]:
# Regras Fuzzy

# Umidade baixa
rule1 = ctrl.Rule(
    umidade['baixa'] & temperatura['baixa'],
    risco_fungos['baixo']
)

rule2 = ctrl.Rule(
    umidade['baixa'] & temperatura['ideal'],
    risco_fungos['baixo']
)

rule3 = ctrl.Rule(
    umidade['baixa'] & temperatura['alta'],
    risco_fungos['baixo']
)

# Umidade média
rule4 = ctrl.Rule(
    umidade['media'] & temperatura['baixa'],
    risco_fungos['baixo']
)

rule5 = ctrl.Rule(
    umidade['media'] & temperatura['ideal'],
    risco_fungos['medio']
)

rule6 = ctrl.Rule(
    umidade['media'] & temperatura['alta'],
    risco_fungos['alto']
)

# Alta umidade
rule7 = ctrl.Rule(
    umidade['alta'] & temperatura['baixa'],
    risco_fungos['medio']
)

rule8 = ctrl.Rule(
    umidade['alta'] & temperatura['ideal'],
    risco_fungos['alto']
)

rule9 = ctrl.Rule(
    umidade['alta'] & temperatura['alta'],
    risco_fungos['alto']
)

In [ ]:
sistema = ctrl.ControlSystem([
    rule1,
    rule2,
    rule3,
    rule4,
    rule5,
    rule6,
    rule7,
    rule8,
    rule9
])

simulador = ctrl.ControlSystemSimulation(sistema)

In [ ]:
def avaliar_risco(valor_umidade, valor_temperatura):

    simulador.input['umidade'] = valor_umidade
    simulador.input['temperatura'] = valor_temperatura

    simulador.compute()

    resultado = simulador.output['risco_fungos']

    print(f'Umidade: {valor_umidade}%')
    print(f'Temperatura: {valor_temperatura} °C')
    print(f'Risco calculado: {resultado:.2f}%')

    if resultado < 35:
        print("Baixo risco de desenvolvimento de fungos.")

    elif resultado < 70:
        print("Risco moderado. Recomenda-se monitoramento.")

    else:
        print("Alto risco. Recomenda-se inspeção e possível aplicação de fungicida.")

In [ ]:
print("Controlador Fuzzy para o Risco de Fungos\n")

valor_umidade = float(input("Digite a umidade do ambiente (0 a 100%): "))
valor_temperatura = float(input("Digite a temperatura (0 a 40 °C): "))

simulador.input['umidade'] = valor_umidade
simulador.input['temperatura'] = valor_temperatura

simulador.compute()

resultado = simulador.output['risco_fungos']

print("\nResultado:")
print(f"Risco de fungos: {resultado:.2f}%")

if resultado < 35:
    print("Classificação: Baixo risco.")
    print("Recomendação: Manter o monitoramento da plantação.")
elif resultado < 70:
    print("Classificação: Risco moderado.")
    print("Recomendação: Intensificar o monitoramento.")
else:
    print("Classificação: Alto risco.")
    print("Recomendação: Avaliar a necessidade de aplicação de fungicida.")

# Exibe o gráfico da saída fuzzy
risco_fungos.view(sim=simulador)